In [4]:
import os, sys
import itertools
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import root_mean_squared_error as rmse
parts=os.getcwd().split('/')
print(parts)
if (parts[-1]=='scripts') |(parts[-1]=='notebooks'):
    parts=parts[:-1]
common_path='/'.join(parts)
srcpath=common_path+'/results/'
out_path=srcpath+'metaanalysis/'
dname_all=['diabetes', 'surgical', 'banking', 'adult', 'criteo','breastcancer']
dname=dname_all[5]
exp_type, solver_type='random', 'sgd'
#respath='%s/%s_%s_unlearning_0.csv'%(srcpath, dname, exp_type)
respath='%s%s/%s_random_unlearn_swish_%s_all.csv'%(srcpath,dname, dname,solver_type)
res_df=pd.read_csv(respath, sep='\t')

code_path='/'.join(parts)+'/scripts/'
sys.path.append(code_path)
from experiment_utils import dataset_health
modelpath='/'.join(parts)+'/models/'
from experiment_utils import data_norm_log
from unlearning.unlearn_eval import *
Xtrain, Ytrain, Xtest, Ytest, features=dataset_health(dname)
zXtrain, zXtest=data_norm_log(Xtrain, Xtest)
print(res_df['n'].unique(), Xtrain.shape)
#zXtrain, zXtest=data_norm_log(Xtrain[features], Xtest[features])
res_df.rename(columns={'num_unlearned_samples':'n_unlearned', 'num_prot':'ppc', 'n':'n_unlearned',
                      'prot_dev_01':'dev_M0M1', 'prot_dev_02':'dev_M0M2', 'prot_dev_12':'dev_M1M2'}, inplace=True)
colnames=list(res_df.columns)
print(len(colnames))
exp_model_specs=[ 'iter','ppc']
model_perf_cols=['et_retrain', 'et_unlearn', 'n_retrain', 'n_unlearned', 'dev_M0M2','dev_M1M2',
               'tr_M0_Bacc', 'tr_M1_Bacc', 'tr_M2_Bacc', #'tr_M0_AUC', 'tr_M1_AUC',  'tr_M2_AUC',
               'tr_M0_nAcc', 'tr_M1_nAcc','tr_retain_corr_M1', 'tr_imp_corr_M1',  'tr_lost_preds_M1',
               'tr_M2_nAcc', 'tr_imp_corr_M2',  'tr_retain_corr_M2','tr_lost_preds_M2' ,
               'te_M0_Bacc', 'te_M1_Bacc', 'te_M2_Bacc', #'te_M0_AUC', 'te_M1_AUC',  'te_M2_AUC',
               'te_M0_nAcc', 'te_M1_nAcc','te_retain_corr_M1', 'te_imp_corr_M1', 'te_lost_preds_M1',
               'te_M2_nAcc', 'te_imp_corr_M2',  'te_retain_corr_M2','te_lost_preds_M2',
               'un_M0_nAcc', 'un_M1_nAcc','un_retain_corr_M1', 'un_imp_corr_M1', 'un_lost_preds_M1',
               'un_M2_nAcc', 'un_imp_corr_M2',  'un_retain_corr_M2','un_lost_preds_M2']
model_param_cols=['et_retrain', 'et_unlearn', 'n_retrain', 'n_unlearned', 
               'dev_M0M1', 'dev_M0M2', 'dev_M1M2' ]
res_df.head(3)
res_df_time=res_df[exp_model_specs+['et_retrain', 'et_unlearn', 'n_retrain', 'n_unlearned']]

['', 'mnt', 'nvme1n1p2', 'codes_datasets', 'toolboxes', 'unlearn-lvq', 'notebooks']
[  1.   5.  20.  30.  50. 100.] (381, 30)
54


In [5]:
#compare_df[compare_df.columns[4:]]=compare_df[compare_df.columns[4:]].astype('float64')
#solver_type='sgd'
#sliced_df=compare_df[exp_model_specs+['n_test']+model_perf_cols].loc[(compare_df['solver_type']==solver_type)]
models, subsets=['M1', 'M2'], ['tr', 'te', 'un']
sliced_df=res_df.copy()#[exp_model_specs+model_param_cols+model_perf_cols].copy()
eval_metrics, nan_counter=[],[]
for subset in subsets:
    if subset==subsets[0]:
        sliced_df['speedupX']=sliced_df['et_retrain'].to_numpy()/sliced_df['et_unlearn'].to_numpy()
        eval_metrics.append('speedupX')
        sliced_df['%s_Acc_M0'%(subset)]=sliced_df['%s_M0_nAcc'%(subset)]/sliced_df['n_retrain']  #'n_test'
        eval_metrics.append('%s_Acc_M0'%(subset))
        eval_metrics.append('dev_M0M2')
        eval_metrics.append('dev_M1M2')
    elif subset=='te':
        sliced_df['%s_Acc_M0'%(subset)]=sliced_df['%s_M0_nAcc'%(subset)]/sliced_df['n_test']  #'n_test'
        eval_metrics.append('%s_Acc_M0'%(subset))
    else:
        sliced_df['%s_Acc_M0'%(subset)]=sliced_df['%s_M0_nAcc'%(subset)]/sliced_df['n_unlearned']  #'n_test'
        eval_metrics.append('%s_Acc_M0'%(subset))
    for model in models:
        if subset=='tr':
            sliced_df['%s_Acc_%s'%(subset, model)]=sliced_df['%s_%s_nAcc'%(subset, model)]/sliced_df['n_retrain']
        elif subset=='te':
            sliced_df['%s_Acc_%s'%(subset, model)]=sliced_df['%s_%s_nAcc'%(subset, model)]/sliced_df['n_test']
        else:
            sliced_df['%s_Acc_%s'%(subset, model)]=sliced_df['%s_%s_nAcc'%(subset, model)]/sliced_df['n_unlearned']
        eval_metrics.append('%s_Acc_%s'%(subset, model))
        if np.sum(sliced_df['%s_M0_nAcc'%subset]==0)==0:
            sliced_df['%s_retain_frac_%s'%(subset, model)]=sliced_df['%s_retain_corr_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
            sliced_df['%s_loss_frac_%s'%(subset, model)]=sliced_df['%s_lost_preds_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
            sliced_df['%s_gain_frac_%s'%(subset, model)]=sliced_df['%s_imp_corr_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
        else:   
            sliced_df['%s_retain_frac_%s'%(subset, model)]=sliced_df['%s_retain_corr_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
            sliced_df['%s_loss_frac_%s'%(subset, model)]=sliced_df['%s_lost_preds_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
            sliced_df['%s_gain_frac_%s'%(subset, model)]=sliced_df['%s_imp_corr_%s'%(subset, model)]/sliced_df['%s_M0_nAcc'%subset]*100
        sliced_df['nan_%s_retain_frac_%s'%(subset, model)]=sliced_df['%s_retain_frac_%s'%(subset, model)].isnull()
        sliced_df['nan_%s_loss_frac_%s'%(subset, model)]=sliced_df['%s_loss_frac_%s'%(subset, model)].isnull()
        sliced_df['nan_%s_gain_frac_%s'%(subset, model)]=sliced_df['%s_gain_frac_%s'%(subset, model)].isnull()
        eval_metrics.append('%s_retain_frac_%s'%(subset, model))
        eval_metrics.append('%s_loss_frac_%s'%(subset, model))
        eval_metrics.append('%s_gain_frac_%s'%(subset, model))
        nan_counter.append('%s_retain_frac_%s'%(subset, model))
        nan_counter.append('%s_loss_frac_%s'%(subset, model))
        nan_counter.append('%s_gain_frac_%s'%(subset, model))
eval_metrics=np.unique(eval_metrics)
nan_cols=np.array(['nan_'+ entry for entry in nan_counter])#nan_man_std=np.intersect1d
nan_res=sliced_df.groupby(['n_unlearned','ppc'])[nan_cols].sum().reset_index()
mean_res=sliced_df.groupby(['n_unlearned','ppc'])[eval_metrics].mean().reset_index()
std_res=sliced_df.groupby(['n_unlearned','ppc'])[eval_metrics].std().reset_index()
col_order=np.array(['n_unlearned', 'ppc', 'speedupX','dev_M0M2','dev_M1M2', 
           'tr_Acc_M0', 'tr_Acc_M1', 'tr_Acc_M2', 'tr_retain_frac_M1', 'tr_retain_frac_M2', 
           'tr_loss_frac_M1', 'tr_loss_frac_M2','tr_gain_frac_M1', 'tr_gain_frac_M2', 
           'te_Acc_M0','te_Acc_M1','te_Acc_M2', 'te_retain_frac_M1', 
           'te_retain_frac_M2','te_loss_frac_M1', 'te_loss_frac_M2', 'te_gain_frac_M1', 'te_gain_frac_M2',
           'un_Acc_M0','un_Acc_M1', 'un_Acc_M2', 'un_retain_frac_M1', 'un_retain_frac_M2',
           'un_loss_frac_M1', 'un_loss_frac_M2', 'un_gain_frac_M1', 'un_gain_frac_M2'])
mean_res=mean_res[col_order].copy()
mean_cols=['mean_'+ entry for entry in eval_metrics]
std_cols=['std_'+ entry for entry in eval_metrics]
mean_res.rename(columns=dict(zip(eval_metrics, mean_cols)), inplace=True)
std_res=std_res[col_order].copy()
std_res.rename(columns=dict(zip(eval_metrics, std_cols)), inplace=True)
common_cols=list(np.intersect1d(list(mean_res.columns), list(std_res.columns)))    
print(common_cols)
final_col_order=list(itertools.chain(*[ [mean_cols[i], std_cols[i]] for i in range(0, len(mean_cols))]))
combined=pd.concat([mean_res, std_res[std_cols]], axis=1)
combined=combined[common_cols+final_col_order]
for nan_col in nan_counter:
    comb_cols=['_'.join(x.split('_')[1:]) for x in list(combined.columns)]
    idx=np.where(nan_col==np.array(comb_cols))[0]
   # print(nan_col, idx, len(comb_cols))
  #  if cout==0:
    insert_loc=idx[0]+2
   # else:
    #    insert_loc=idx[0]+2
    combined.insert(insert_loc,'nan_%s'%nan_col , nan_res['nan_%s'%nan_col])
metapath='%s%s/%s_%s_summary_swish_all.csv'%(out_path,solver_type, dname, exp_type)
print(metapath)
combined.to_csv(metapath, sep='\t', index=False)

['n_unlearned', 'ppc']
/mnt/nvme1n1p2/codes_datasets/toolboxes/unlearn-lvq/results/metaanalysis/sgd/breastcancer_random_summary_swish_all.csv


In [3]:
combined.loc[combined['n_unlearned']==1]
del combined

In [48]:
#sliced_df[nan_cols]
combined[combined.columns[5:]]#.groupby(['n_unlearned','ppc']).sum().reset_index()

,std_dev_M1M2,mean_speedupX,std_speedupX,mean_te_Acc_M0,std_te_Acc_M0,mean_te_Acc_M1,std_te_Acc_M1,mean_te_Acc_M2,std_te_Acc_M2,mean_te_gain_frac_M1,...,nan_un_loss_frac_M1,mean_un_loss_frac_M2,std_un_loss_frac_M2,nan_un_loss_frac_M2,mean_un_retain_frac_M1,std_un_retain_frac_M1,nan_un_retain_frac_M1,mean_un_retain_frac_M2,std_un_retain_frac_M2,nan_un_retain_frac_M2
0,1.142283,1092.725738,269.847219,0.898276,0.0,0.895679,0.000066,0.714275,0.162770,0.902703,...,0,30.000000,29.814240,0,100.000000,0.000000,0,70.000000,29.814240,0
1,0.928730,879.535823,86.793487,0.898762,0.0,0.895387,0.000102,0.818354,0.152383,0.415991,...,0,5.000000,11.180340,0,100.000000,0.000000,0,95.000000,11.180340,0
2,0.088101,746.735089,148.451474,0.898276,0.0,0.895703,0.000102,0.702913,0.061839,0.908108,...,0,19.994438,6.085527,0,99.310345,1.542116,0,80.005562,6.085527,0
3,0.123079,512.162709,46.245907,0.898762,0.0,0.895387,0.000102,0.638432,0.070621,0.418693,...,0,32.932631,15.067224,0,99.285714,1.597191,0,67.067369,15.067224,0
4,0.095836,747.277670,76.540737,0.898276,0.0,0.895703,0.000054,0.691333,0.052981,0.894595,...,0,27.402357,7.514331,0,99.189086,0.399521,0,72.597643,7.514331,0
5,0.031377,494.056069,35.336083,0.898762,0.0,0.895387,0.000054,0.729886,0.081269,0.407888,...,0,20.244477,10.521314,0,99.392782,0.602672,0,79.755523,10.521314,0
6,0.008880,551.688802,34.291673,0.898276,0.0,0.896164,0.000203,0.733382,0.016030,0.859459,...,0,23.846475,1.773839,0,98.605077,0.173819,0,76.153525,1.773839,0
7,0.055691,376.722751,16.506961,0.898762,0.0,0.895363,0.000000,0.647172,0.006953,0.405186,...,0,30.112657,1.676038,0,99.398869,0.110124,0,69.887343,1.676038,0
